# Task 1: Symbolic Unconditioned Music Generation

**Goal:** Train a small causal Transformer on the MAESTRO v3 piano MIDI dataset to learn a
distribution over note-event token sequences, then sample novel pieces from it.

**Pipeline:**
```
MAESTRO MIDIs → REMI tokenize → 512-token chunks
  → nn.TransformerEncoder (causal) → cross-entropy
  → autoregressive sampling → symbolic_unconditioned.mid
```

**Sections:**
1. Exploratory Analysis, Data Collection & Pre-processing
2. Modelling
3. Evaluation
4. Related Work

In [ ]:
import os, sys, json, random, math, collections, zipfile, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from miditok import REMI

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 1. Exploratory Analysis, Data Collection & Pre-processing

### 1.1 Dataset Context

**MAESTRO v3** (Music And Audio Recorded in Tight Synchrony) contains ~200 hours of
virtuosic piano performances with aligned MIDI and audio, recorded at the International
Piano-e-Competition over 14 years (Hawthorne et al., 2019).
We use only the **MIDI files** (~57 MB) for symbolic modelling.
The dataset ships with an official `train / validation / test` split by performer
to avoid data leakage.

**Why MAESTRO?**
- Single-instrument (piano) → compact vocabulary, coherent style.
- High-quality, performance-level dynamics and articulation.
- Official splits → reproducible evaluation.

In [ ]:
DATA_DIR = Path("data/maestro-v3.0.0")
CSV_PATH = DATA_DIR / "maestro-v3.0.0.csv"
ZIP_PATH = Path("data/maestro-v3.0.0-midi.zip")
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
GEN_DIR  = Path("generated");   GEN_DIR.mkdir(exist_ok=True)

MAESTRO_URL = (
    "https://storage.googleapis.com/magentadata/datasets/maestro/"
    "v3.0.0/maestro-v3.0.0-midi.zip"
)

if not CSV_PATH.exists():
    Path("data").mkdir(exist_ok=True)
    print("Downloading MAESTRO v3 MIDI-only (~57 MB) ...")
    urllib.request.urlretrieve(MAESTRO_URL, ZIP_PATH)
    print("Extracting ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("data/")
    print("Done.")
else:
    print("MAESTRO already present.")

df = pd.read_csv(CSV_PATH)
display(df.head(3))
print(f"\nTotal entries : {len(df)}")
print(f"\nSplit distribution:\n{df['split'].value_counts().to_string()}")

In [ ]:
print("Duration (seconds) statistics per split:")
display(df.groupby("split")["duration"].describe().round(2))

print("\nYear distribution:")
display(df.groupby(["year", "split"]).size().unstack(fill_value=0))

### 1.2 Tokenisation (REMI scheme)

We use the **REMI** (Revamped MIDI-derived events) representation
(Huang & Yang, 2020) from the `miditok` library.

REMI encodes each piece as a flat sequence of discrete events:
`Bar`, `Position`, `Pitch`, `Velocity`, `Duration` — each with its own
integer id drawn from a shared fixed vocabulary.

**Advantages over dense piano-roll:**
- Compact sequences (~1–3k tokens per piece vs. tens of thousands of frames).
- Explicit rhythmic structure via `Bar` / `Position` tokens.
- Losslessly invertible back to MIDI.

In [ ]:
TOKENIZER = REMI()
VOCAB_SIZE = len(TOKENIZER.vocab)
PAD_ID = TOKENIZER["PAD_None"]
BOS_ID = TOKENIZER["BOS_None"]
EOS_ID = TOKENIZER["EOS_None"]

print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"PAD={PAD_ID}  BOS={BOS_ID}  EOS={EOS_ID}")
print("\nFirst 20 tokens:")
for tok, idx in list(TOKENIZER.vocab.items())[:20]:
    print(f"  {idx:3d}  {tok}")

In [ ]:
# Sanity-check tokenisation on one file
example_row  = df[df["split"] == "train"].iloc[0]
example_path = DATA_DIR / example_row["midi_filename"]

tok_seq = TOKENIZER.encode(example_path)
ids     = tok_seq.ids
print(f"File           : {example_path.name}")
print(f"Token count    : {len(ids)}")
print(f"First 30 ids   : {ids[:30]}")
print(f"First 30 names : {[TOKENIZER[i] for i in ids[:30]]}")

### 1.3 Full Tokenisation & Chunking

Each MIDI file is tokenised to a 1-D integer sequence.
Long sequences are cut into **512-token windows** with **256-token stride**
(50% overlap) so the model sees phrase boundaries in both halves of each window.
Short sequences are padded to 512 with `PAD` tokens (masked during training).

In [ ]:
SEQ_LEN = 512
STRIDE  = 256

def tokenise_split(split_name, max_files=None):
    rows = df[df["split"] == split_name]
    if max_files:
        rows = rows.head(max_files)
    all_chunks, all_raw_lens = [], []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=split_name):
        path = DATA_DIR / row["midi_filename"]
        try:
            ids = TOKENIZER.encode(path).ids
        except Exception:
            continue
        all_raw_lens.append(len(ids))
        for start in range(0, max(1, len(ids) - SEQ_LEN + 1), STRIDE):
            chunk = ids[start : start + SEQ_LEN]
            if len(chunk) < SEQ_LEN:
                chunk = chunk + [PAD_ID] * (SEQ_LEN - len(chunk))
            all_chunks.append(chunk)
    return all_chunks, all_raw_lens

print("Tokenising training split ...")
train_chunks, train_lens = tokenise_split("train")
print(f"  chunks : {len(train_chunks):,}")

print("Tokenising validation split ...")
val_chunks, val_lens = tokenise_split("validation")
print(f"  chunks : {len(val_chunks):,}")

print("Tokenising test split ...")
test_chunks, test_lens = tokenise_split("test")
print(f"  chunks : {len(test_chunks):,}")

### 1.4 EDA Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("MAESTRO v3 - Exploratory Analysis", fontsize=14)

# (a) Token sequence length per file
ax = axes[0]
all_lens = train_lens + val_lens + test_lens
ax.hist(all_lens, bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("Token sequence length per file")
ax.set_ylabel("Count")
ax.set_title("(a) Token lengths per file")
ax.axvline(np.median(all_lens), color="red", linestyle="--",
           label=f"Median={int(np.median(all_lens))}")
ax.legend()

# (b) Top-30 token frequency (train, first 5000 chunks)
ax = axes[1]
flat = [t for chunk in train_chunks[:5000] for t in chunk if t != PAD_ID]
counter = collections.Counter(flat)
top30 = counter.most_common(30)
labels = [TOKENIZER[i] for i, _ in top30]
counts = [c for _, c in top30]
ax.barh(range(30), counts[::-1], color="steelblue")
ax.set_yticks(range(30))
ax.set_yticklabels(labels[::-1], fontsize=7)
ax.set_xlabel("Count")
ax.set_title("(b) Top-30 token frequency (train)")

# (c) Pitch distribution
ax = axes[2]
pitch_ids = {v: int(k.split("_")[1])
             for k, v in TOKENIZER.vocab.items() if k.startswith("Pitch_")}
pitch_counts = collections.Counter(
    {pitch_ids[i]: counter[i] for i in pitch_ids if i in counter})
pitches = sorted(pitch_counts.keys())
ax.bar(pitches, [pitch_counts[p] for p in pitches], color="salmon", width=1.0)
ax.set_xlabel("MIDI pitch (21=A0 ... 108=C8)")
ax.set_ylabel("Count")
ax.set_title("(c) Pitch distribution (train)")

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nDataset summary:")
print(f"  Train files  : {len(df[df.split=='train'])}")
print(f"  Val   files  : {len(df[df.split=='validation'])}")
print(f"  Test  files  : {len(df[df.split=='test'])}")
print(f"  Train chunks : {len(train_chunks):,}")
print(f"  Val   chunks : {len(val_chunks):,}")
print(f"  Test  chunks : {len(test_chunks):,}")
print(f"  Vocab size   : {VOCAB_SIZE}")
print(f"  Chunk length : {SEQ_LEN}  Stride : {STRIDE}")

---
## 2. Modelling

### 2.1 Dataset & DataLoader

Each 512-token chunk becomes one training example.
The model is trained with **next-token prediction**: input = `chunk[:-1]`, target = `chunk[1:]`.

In [ ]:
class MaestroDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = [torch.tensor(c, dtype=torch.long) for c in chunks]

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        seq = self.chunks[idx]
        return seq[:-1], seq[1:]   # input, target — both length SEQ_LEN-1

BATCH_SIZE = 64

train_ds = MaestroDataset(train_chunks)
val_ds   = MaestroDataset(val_chunks)
test_ds  = MaestroDataset(test_chunks)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda"))

x, y = next(iter(train_loader))
print(f"Input  shape : {x.shape}   dtype: {x.dtype}")
print(f"Target shape : {y.shape}   dtype: {y.dtype}")
print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

### 2.2 Model Architecture

**Formulation:** Unconditioned symbolic generation = language modelling over token sequences.
We minimise cross-entropy loss on next-token prediction:

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log p_{\theta}(x_t \mid x_{<t})$$

**Architecture:** GPT-style decoder using PyTorch's `nn.TransformerEncoder` with a
**causal (upper-triangular) mask** so position $t$ cannot attend to future positions.

```
tokens (B, T)
  -> Embedding(vocab_size, d_model) + SinusoidalPositionalEncoding
  -> TransformerEncoder [4 layers, d_model=256, 8 heads, ffn=1024, Pre-LN]
       with causal mask (upper-tri = -inf)
  -> Linear(d_model, vocab_size)
  -> cross-entropy with next token
```

**Why `nn.TransformerEncoder` and not `nn.Transformer`?**
`nn.Transformer` is designed for seq2seq (encoder + decoder, two input sequences).
Pure language modelling only needs one sequence with causal masking —
`nn.TransformerEncoder` with a triangular attention mask achieves exactly that,
with less boilerplate.

| Hyperparameter | Value | Reasoning |
|---|---|---|
| `d_model` | 256 | Fits 4080 VRAM easily at batch 64 |
| `nhead` | 8 | 32-dim per head — standard ratio |
| `num_layers` | 4 | ~10M params; trains in hours |
| `dim_feedforward` | 1024 | 4x d_model — standard |
| `dropout` | 0.1 | Light regularisation |

In [ ]:
class SinusoidalPE(nn.Module):
    """Fixed sinusoidal positional encoding."""
    def __init__(self, d_model, max_len=1024, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class MusicLM(nn.Module):
    """Causal language model for symbolic music generation."""

    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4,
                 dim_feedforward=1024, dropout=0.1, max_len=1024):
        super().__init__()
        self.d_model = d_model
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_enc = SinusoidalPE(d_model, max_len, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
            norm_first=True,          # Pre-LN for stable training
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            enable_nested_tensor=False)
        self.head = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.embed.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
        nn.init.normal_(self.head.weight, std=0.02)

    @staticmethod
    def make_causal_mask(seq_len, device):
        """Upper-triangular mask: position t cannot see t+1 ... T."""
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.masked_fill(mask.bool(), float("-inf"))

    def forward(self, x, pad_mask=None):
        """
        x        : (B, T) long
        pad_mask : (B, T) bool, True where padded
        returns logits (B, T, vocab_size)
        """
        T      = x.size(1)
        causal = self.make_causal_mask(T, x.device)
        emb    = self.pos_enc(self.embed(x) * math.sqrt(self.d_model))
        out    = self.transformer(emb, mask=causal,
                                  src_key_padding_mask=pad_mask,
                                  is_causal=True)
        return self.head(out)


D_MODEL    = 256
NHEAD      = 8
NUM_LAYERS = 4
FFN_DIM    = 1024
DROPOUT    = 0.1

model = MusicLM(VOCAB_SIZE, D_MODEL, NHEAD, NUM_LAYERS, FFN_DIM, DROPOUT).to(DEVICE)

total  = sum(p.numel() for p in model.parameters())
train_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {train_:,}")
print(model)

### 2.3 Training

**Objective:** minimise cross-entropy between model predictions and next-token targets,
ignoring `PAD` positions.

**Optimisation choices:**
- **AdamW** with weight decay 0.01 — standard for transformers; decouples L2 from adaptive momentum.
- **Linear warmup** (2,000 steps) then cosine decay — stabilises early training.
- **Mixed precision (fp16)** via `torch.cuda.amp` — roughly 2x throughput on the RTX 4080.
- **Early stopping** (patience = 5 epochs) on validation perplexity.

In [ ]:
def get_pad_mask(x):
    """True where tokens are PAD (ignored in attention)."""
    return x == PAD_ID   # (B, T)

def compute_loss(model, x, y):
    pad_mask = get_pad_mask(x)
    logits   = model(x, pad_mask)         # (B, T, V)
    return F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        y.reshape(-1),
        ignore_index=PAD_ID,
    )

def evaluate(model, loader, max_batches=None):
    model.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if max_batches and i >= max_batches:
                break
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = compute_loss(model, x, y)
            total_loss += loss.item(); n += 1
    model.train()
    return total_loss / max(n, 1)

def get_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.05, 0.5 * (1 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
LR           = 1e-4
WEIGHT_DECAY = 1e-2
NUM_EPOCHS   = 30
PATIENCE     = 5
WARMUP_STEPS = 2000
LOG_INTERVAL = 200   # log every N batches

USE_AMP   = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.bfloat16

if not USE_AMP:
    print("WARNING: CUDA not available — training on CPU (much slower).")
    print("Update your NVIDIA driver at https://www.nvidia.com/Download/index.aspx")
    print("to enable GPU acceleration with PyTorch 2.12.\n")

optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler      = GradScaler(enabled=USE_AMP)
total_steps = NUM_EPOCHS * len(train_loader)
scheduler   = get_scheduler(optimizer, WARMUP_STEPS, total_steps)

best_val_loss  = float("inf")
patience_count = 0
history        = {"train_loss": [], "val_loss": [], "val_ppl": []}

print(f"Training for up to {NUM_EPOCHS} epochs")
print(f"Batches per epoch : {len(train_loader)}")
print(f"Warmup steps      : {WARMUP_STEPS}  /  Total steps : {total_steps}\n")

global_step = 0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss, n_batches = 0.0, 0

    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = compute_loss(model, x, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss  += loss.item()
        n_batches   += 1
        global_step += 1

        if (batch_idx + 1) % LOG_INTERVAL == 0:
            avg    = epoch_loss / n_batches
            lr_now = scheduler.get_last_lr()[0]
            print(f"  Epoch {epoch:2d} | step {global_step:6d} | "
                  f"train_loss={avg:.4f} | lr={lr_now:.2e}")

    avg_train = epoch_loss / n_batches
    val_loss  = evaluate(model, val_loader, max_batches=200)
    val_ppl   = math.exp(val_loss)

    history["train_loss"].append(avg_train)
    history["val_loss"].append(val_loss)
    history["val_ppl"].append(val_ppl)

    print(f"Epoch {epoch:2d} | train={avg_train:.4f} | "
          f"val={val_loss:.4f} | ppl={val_ppl:.1f}")

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        patience_count = 0
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_ppl": val_ppl,
            "history": history,
        }, CKPT_DIR / "best_model.pt")
        print(f"  -> Saved best checkpoint (val_ppl={val_ppl:.1f})")
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

print(f"\nBest validation perplexity : {math.exp(best_val_loss):.2f}")

---
## 3. Evaluation

### 3.1 Baseline: Bigram Markov Model

As a baseline we train a **bigram language model** on the same REMI token sequences.
It predicts $p(x_t \mid x_{t-1})$ using maximum-likelihood counts with add-one (Laplace) smoothing.

This is equivalent to the Markov chain models from Module 3.
If our transformer cannot beat this, something is wrong.

In [ ]:
class BigramLM:
    """Add-one smoothed bigram language model over REMI token ids."""

    def __init__(self, vocab_size, pad_id):
        self.V      = vocab_size
        self.pad_id = pad_id
        self.counts  = collections.defaultdict(lambda: collections.defaultdict(int))
        self.unigram = collections.defaultdict(int)

    def train(self, chunks):
        for chunk in tqdm(chunks, desc="Training bigram"):
            prev = None
            for tok in chunk:
                if tok == self.pad_id:
                    prev = None; continue
                self.unigram[tok] += 1
                if prev is not None:
                    self.counts[prev][tok] += 1
                prev = tok

    def log_prob(self, context, next_tok):
        """log p(next_tok | context) with add-one smoothing."""
        denom = sum(self.counts[context].values()) + self.V
        numer = self.counts[context].get(next_tok, 0) + 1
        return math.log(numer / denom)

    def perplexity(self, chunks, max_chunks=5000):
        total_ll, total_n = 0.0, 0
        for chunk in chunks[:max_chunks]:
            prev = None
            for tok in chunk:
                if tok == self.pad_id:
                    prev = None; continue
                if prev is not None:
                    total_ll += self.log_prob(prev, tok)
                    total_n  += 1
                prev = tok
        return math.exp(-total_ll / max(total_n, 1))


bigram = BigramLM(VOCAB_SIZE, PAD_ID)
bigram.train(train_chunks)

bigram_val_ppl  = bigram.perplexity(val_chunks,  max_chunks=3000)
bigram_test_ppl = bigram.perplexity(test_chunks, max_chunks=3000)
print(f"Bigram val  perplexity : {bigram_val_ppl:.2f}")
print(f"Bigram test perplexity : {bigram_test_ppl:.2f}")

### 3.2 Load Best Checkpoint & Compute Test Perplexity

In [ ]:
ckpt = torch.load(CKPT_DIR / "best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
history = ckpt["history"]
print(f"Loaded best checkpoint from epoch {ckpt['epoch']}  "
      f"(val_ppl={ckpt['val_ppl']:.2f})")

test_loss            = evaluate(model, test_loader)
transformer_test_ppl = math.exp(test_loss)
best_val_loss        = ckpt["val_loss"]

print(f"\nTransformer test perplexity : {transformer_test_ppl:.2f}")
print(f"Bigram      test perplexity : {bigram_test_ppl:.2f}")
print(f"\nImprovement over bigram : {bigram_test_ppl / transformer_test_ppl:.1f}x")

### 3.3 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Transformer Training Curves", fontsize=14)

epochs_ax = range(1, len(history["train_loss"]) + 1)

ax = axes[0]
ax.plot(epochs_ax, history["train_loss"], label="Train loss", marker="o", markersize=3)
ax.plot(epochs_ax, history["val_loss"],   label="Val loss",   marker="s", markersize=3)
ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-entropy loss")
ax.set_title("Loss curves"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(epochs_ax, history["val_ppl"], color="darkorange", marker="s", markersize=3,
        label="Transformer val PPL")
ax.axhline(bigram_val_ppl, color="red", linestyle="--",
           label=f"Bigram baseline ({bigram_val_ppl:.1f})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Perplexity")
ax.set_title("Validation Perplexity vs. Bigram Baseline")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary table
summary = pd.DataFrame({
    "Model":      ["Bigram (baseline)", "Transformer (ours)"],
    "Val PPL":    [round(bigram_val_ppl,  1), round(math.exp(best_val_loss), 1)],
    "Test PPL":   [round(bigram_test_ppl, 1), round(transformer_test_ppl,   1)],
})
display(summary)

### 3.4 Evaluation Discussion

**Quantitative metric — Perplexity:**
Perplexity $\text{PPL} = e^{\mathcal{L}}$ measures how surprised the model is by
held-out tokens. Lower is better.

**Caveats about perplexity and musical quality:**
- Perplexity measures probabilistic fit to the data distribution, not musical aesthetics.
- A model with lower perplexity generally produces more stylistically consistent tokens —
  common pitches, reasonable durations — but does not guarantee good phrasing or harmony.
- Subjective listenability and harmonic coherence are separate properties, evaluated
  by listening to generated samples (Section 4).

**Baselines discussion:**
- The bigram model (our baseline) uses only one token of history. It can learn which
  tokens commonly follow others (e.g. `Pitch_X` often followed by `Velocity_Y`) but
  has no sense of phrase, key, or bar-level structure.
- Our transformer, with 511 tokens of context, can in principle model local harmonic
  progressions, rhythmic patterns, and phrase repetition.
- Beating the bigram by a large margin is expected; stronger baselines would include
  a 4-gram model with Kneser-Ney smoothing or a hidden Markov model.

---
## 4. Generation

### 4.1 Autoregressive Sampler

We use **temperature-scaled top-k sampling**:
1. Start with a short primer (first 32 tokens from a validation piece).
2. Run the model forward to get logits over the next token.
3. Divide logits by temperature $\tau$ (lower = more deterministic).
4. Keep only the top-$k$ logits; set the rest to $-\infty$.
5. Sample from the resulting categorical distribution.
6. Append the sample and repeat until EOS or `max_new_tokens`.

In [ ]:
@torch.no_grad()
def generate(model, primer_ids, max_new_tokens=512, temperature=0.9,
             top_k=50, device=DEVICE):
    model.eval()
    generated = list(primer_ids)

    for _ in range(max_new_tokens):
        ctx    = generated[-SEQ_LEN:]
        x      = torch.tensor([ctx], dtype=torch.long, device=device)
        pad    = get_pad_mask(x)
        logits = model(x, pad)[0, -1, :]   # (vocab,) — last position

        logits = logits / max(temperature, 1e-6)

        if top_k > 0:
            topk_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < topk_vals[-1]] = float("-inf")

        probs    = torch.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_tok)

        if next_tok == EOS_ID:
            break

    return generated

In [ ]:
import shutil

val_rows = df[df["split"] == "validation"].reset_index(drop=True)
generated_files = []

for i in range(5):
    primer_path = DATA_DIR / val_rows.iloc[i]["midi_filename"]
    primer_ids  = TOKENIZER.encode(primer_path).ids[:32]

    gen_ids  = generate(model, primer_ids, max_new_tokens=512,
                        temperature=0.9, top_k=50)
    out_path = GEN_DIR / f"sample_{i+1}.mid"

    try:
        score = TOKENIZER.decode(gen_ids)
        score.dump_midi(str(out_path))
        generated_files.append(out_path)
        print(f"Saved : {out_path}  ({len(gen_ids)} tokens)")
    except Exception as e:
        print(f"  Sample {i+1}: MIDI write error — {e}")

if generated_files:
    shutil.copy(generated_files[0], "symbolic_unconditioned.mid")
    print("\nsymbolic_unconditioned.mid  <-  sample_1.mid")

In [ ]:
def token_stats(ids):
    """Quick structural stats for a generated token sequence."""
    pitch_ids = {v: int(k.split("_")[1])
                 for k, v in TOKENIZER.vocab.items() if k.startswith("Pitch_")}
    pitches   = [pitch_ids[t] for t in ids if t in pitch_ids]
    bar_count = sum(1 for t in ids if TOKENIZER.vocab.get("Bar_None") == t)
    return {
        "total_tokens":  len(ids),
        "note_count":    len(pitches),
        "unique_pitches":len(set(pitches)),
        "pitch_range":   (min(pitches, default=0), max(pitches, default=0)),
        "approx_bars":   bar_count,
    }

print("Generated sample statistics:")
for i, path in enumerate(generated_files, 1):
    primer_path = DATA_DIR / val_rows.iloc[i-1]["midi_filename"]
    primer_ids  = TOKENIZER.encode(primer_path).ids[:32]
    gen_ids_tmp = generate(model, primer_ids, max_new_tokens=512,
                           temperature=0.9, top_k=50)
    s = token_stats(gen_ids_tmp)
    print(f"  Sample {i}: {s}")

---
## 5. Discussion of Related Work

### The MAESTRO Dataset
Hawthorne et al. (2019) introduced **MAESTRO** primarily for automatic music transcription.
It has since become the standard benchmark for symbolic piano generation,
used in Music Transformer, PianoBART, and many others.
We exploit its official performer-based splits to ensure no data leakage between train/val/test.

**Reference:** Hawthorne, C., Stasyuk, A., Roberts, A., Simon, I., Huang, C. Z. A.,
Dieleman, S., Eck, D. (2019). *Enabling Factorized Piano Music Modeling and Generation
with the MAESTRO Dataset.* ICLR 2019.

---

### Symbolic Transformers for Music

**Music Transformer** (Huang et al., 2018) was among the first to apply self-attention
to symbolic piano music at scale.  They introduced *relative positional self-attention*
to capture repeating motifs at a distance — a weakness of absolute encodings like ours.
Their model was trained on MAESTRO v1 and generated multi-minute coherent pieces.
Our simpler sinusoidal absolute encoding is less expressive for long-range structure,
but is sufficient to demonstrate the pipeline at course scale.

**Reference:** Huang, C. Z. A., Vaswani, A., Uszkoreit, J., Simon, I., Hawthorne, C.,
Shazeer, N., Parmar, N., Eck, D. (2018). *Music Transformer: Generating Music with
Long-term Structure.* ICLR 2019.

---

### REMI Tokenisation
Huang & Yang (2020) proposed REMI, arguing that explicitly encoding bar and beat positions
(rather than only time-delta events used in MIDI-like representations) helps models learn
metric regularity. They showed REMI-trained transformers produce more rhythmically coherent
output than MIDI-event-based ones on pop piano data.

**Reference:** Huang, Y. S. & Yang, Y. H. (2020). *Pop Music Transformer: Beat-based
Modeling and Generation of Expressive Pop Piano Compositions.* ACM MM 2020.

---

### How Our Results Compare
Small REMI-based piano LMs typically achieve validation perplexities in the range of 5–15
(depending on model size and training data). Our transformer should substantially beat the
bigram baseline (~50–200) and approach the range reported for similar small-scale models.
The Music Transformer with relative attention achieves lower perplexity and better long-range
coherence, but requires significantly more compute. Our goal is a principled pedagogical
demonstration, not SOTA.

### Limitations
- **Absolute positional encoding** limits long-range form (ABA, variations).
- **No conditioning** on key, tempo, or style.
- **Fixed-length chunks** lose cross-chunk dependencies for pieces > 512 tokens.

---
## Appendix: Submission Files

Run the cell below to confirm all required files are present before exporting to HTML.

| File | Description |
|---|---|
| `symbolic_unconditioned.mid` | Generated piano piece (primary submission file) |
| `workbook.html` | This notebook exported as HTML |
| `eda_plots.png` | EDA figure |
| `training_curves.png` | Loss / perplexity training curves |

In [ ]:
print("Submission file checklist:")
required = ["symbolic_unconditioned.mid", "eda_plots.png", "training_curves.png"]
for f in required:
    exists = Path(f).exists()
    size   = Path(f).stat().st_size / 1024 if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}]  {f:<40}  {size:8.1f} KB")

print("\nTo export to HTML, run in terminal:")
print("  jupyter nbconvert --to html --no-input workbook.ipynb --output workbook.html")